[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/tracing/langchain_agent_tracing_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">追踪与评估 LangChain OpenAI 智能体应用</h1>

借助支持函数调用的 OpenAI API，构建自己的智能体变得更加容易。

本 Notebook 教程展示如何用不到 50 行代码编写 OpenAI 智能体，并使用 Phoenix 检查其内部执行过程。实现虽精简，但具备完整功能，包括持续对话和使用工具。

## 1. 安装依赖并导入库

安装 Phoenix、LangChain 和 OpenAI。

In [ ]:
!pip install langchain langchain-openai langgraph arize-phoenix openinference-instrumentation-langchain

导入库。

In [ ]:
import os
from getpass import getpass

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openinference.instrumentation.langchain import LangChainInstrumentor
from phoenix.otel import register

import phoenix as px

## 2. 启动 Phoenix

可以在后台运行 Phoenix，收集已通过 `OpenInferenceTracer` 添加埋点的 LangChain 应用发出的追踪数据。

启动 Phoenix，按照单元输出中的说明打开 Phoenix 界面。由于尚未运行 LangChain 应用，此时界面应为空。

In [ ]:
(session := px.launch_app()).view()

## 3. 配置 OpenAI 凭证

确认已设置 OpenAI 凭证。

In [ ]:
if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥： ")
os.environ["OPENAI_API_KEY"] = openai_api_key

## 4. 构建智能体并添加埋点

现在为 LangChain 添加埋点，将追踪数据发送到 Phoenix。

In [ ]:
tracer_provider = register()
LangChainInstrumentor(tracer_provider=tracer_provider).instrument(skip_dep_check=True)

定义智能体要使用的 LLM 模型。

In [ ]:
llm = ChatOpenAI(temperature=0)

定义 LLM 可以调用的工具。本例使用以下工具：

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


@tool
def calculator(expression: str) -> str:
    """计算数学表达式。"""
    return str(eval(expression))


tools = [calculator]

现在定义一个能够回答问题并调用工具的智能体。

核心逻辑分为三步：

- 调用 OpenAI，决定是否调用工具、调用哪个工具，以及使用什么参数。
- 使用这些参数调用工具，获得输出。
- 调用 OpenAI，根据对话上下文和工具输出生成回答。

接下来初始化智能体。

In [ ]:
agent_executor = create_agent(llm, tools)

## 5. 与智能体对话

现在和智能体聊聊！调用智能体时，`OpenInferenceTracer` 会将执行追踪记录到 Phoenix。

In [ ]:
response = agent_executor.invoke({"messages": [("user", "What is 47 raised to the 5th power?")]})
response

再聊几轮，这次提出一些追问。

In [ ]:
queries = [
    "What is (121 * 3) + 42?",
    "what is 3 * 3?",
    "what is 4 * 4?",
    "what is 75 * (3 + 4)?",
    "what is 23 times 87",
]

for query in queries:
    print(f"> {query}")
    response = agent_executor.invoke({"messages": [("user", query)]})
    print(response)
    print("---")

在浏览器中打开 `session.url`，查看 Phoenix 中的追踪。注意，LLM span 包含 OpenAI 函数调用，可以据此查看 LLM 根据查询选择了哪个工具。

有关函数调用的更多信息，请参阅 [OpenAI API 文档](https://openai.com/blog/function-calling-and-other-api-updates)。


In [ ]:
print(f"在 Phoenix 中查看追踪： {session.url}")

## 6. 导出追踪数据

可以将追踪数据导出为 pandas dataframe，以便进一步分析和评估。

In [ ]:
from phoenix.client import Client

trace_df = Client().spans.get_spans_dataframe()
trace_df